# Wan 2.1 on a T4: what makes a clip slow?

First spike: 25.5 s per denoising step, 15 min for a 3 s clip, 39 min for 5 s. This notebook times
**5 steps** of several variants (about 4 minutes each) and prints seconds per step and peak VRAM.

Run the cells **one at a time with Shift+Enter, top to bottom** (not "Run all"; the last two are optional).
Same settings as before: Accelerator **GPU T4 x2**, Internet **On**. Free, nothing on AWS.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one T4, like the real instance (must be set before torch loads)
import json, shutil, subprocess, sys, time

REPORT = []
def check(name, status, detail=""):
    """status: PASS, FAIL or INFO"""
    REPORT.append({"check": name, "status": status, "detail": detail})
    print(f"[{status}] {name}: {detail}")

import psutil, torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
vram = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
check("GPU is a T4", "PASS" if "T4" in gpu else "FAIL", f"{gpu}, {vram:.1f} GiB VRAM")
check("RAM", "INFO", f"{psutil.virtual_memory().total / 2**30:.1f} GiB total (g4dn.2xlarge has 32)")
free_tmp = shutil.disk_usage("/tmp").free / 2**30
check("Disk for weights in /tmp", "PASS" if free_tmp > 40 else "FAIL", f"{free_tmp:.0f} GiB free, need about 35")
check("torch", "INFO", torch.__version__ + " (cuda " + str(torch.version.cuda) + "); the image will carry torch 2.8.0")


In [ ]:
PINS = ['diffusers==0.35.2', 'transformers==4.57.6', 'accelerate==1.15.0', 'safetensors==0.8.0', 'sentencepiece==0.2.2', 'ftfy==6.3.1', 'regex==2026.9.29', 'imageio==2.38.1', 'imageio-ffmpeg==0.6.0']
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
import importlib.metadata as md
got = {p: md.version(p) for p in ("diffusers", "transformers", "accelerate", "safetensors", "sentencepiece", "ftfy", "imageio", "imageio-ffmpeg")}
check("pinned packages installed", "PASS" if r.returncode == 0 else "FAIL", json.dumps(got))


In [ ]:
# The worker's own files, embedded by scripts/make_spike_notebook.py
import pathlib
FILES = {'__init__.py': '"""The GPU worker: takes jobs off the queue, runs the model, uploads the clip (spec section 4)."""\n', 'errors.py': '"""The ways a job attempt can end other than success."""\n\n\nclass Aborted(Exception):\n    """The generator stopped early because `should_abort()` turned true."""\n\n\nclass BadMessage(Exception):\n    """A queue message that cannot be tied to a job (no usable job_id or sub). Nobody to tell."""\n\n\nclass PermanentJobError(Exception):\n    """The job can never succeed. Becomes a `.failed` marker the user sees, so `message` must be\n    safe to show: no paths, no stack traces."""\n\n    def __init__(self, code, message, field=None):\n        super().__init__(message)\n        self.code = code\n        self.message = message\n        self.field = field\n\n    def to_json(self):\n        out = {"code": self.code}\n        if self.field:\n            out["field"] = self.field\n        out["message"] = self.message\n        return out\n', 'wan.py': '"""Wan 2.1 T2V 1.3B through diffusers\' WanPipeline (spec section 4, "Model").\n\nUNVERIFIED: nothing here has run on a GPU yet. The Kaggle spike runs this same class and checks\nspeed, memory, the step count and whether fp16 gives clean frames. Until then treat the numbers\nbelow (steps, guidance, flow_shift) as the documented defaults, not measured ones.\n\ntorch and diffusers are imported inside methods, so the rest of the worker (and its tests)\nload without them.\n"""\nimport gc\nimport pathlib\nimport random\n\nfrom .errors import Aborted\n\nWIDTH, HEIGHT, FPS = 832, 480, 16\nGUIDANCE_SCALE = 5.0\nFLOW_SHIFT = 3.0          # diffusers\' Wan docs: 3.0 for 480p, 5.0 for 720p\nFRAMES = {"3": 49, "5": 81}   # 16 fps, and the model needs 4n+1 frames\nMAX_SEED = 2147483647\n\n\ndef num_frames(duration_seconds):\n    return FRAMES[duration_seconds]\n\n\ndef pick_seed(params, rng=random):\n    seed = params.get("seed")\n    return seed if seed is not None else rng.randint(0, MAX_SEED)\n\n\nclass WanGenerator:\n    def __init__(self, models_dir, model_name, steps):\n        self.path = str(pathlib.Path(models_dir) / model_name)\n        self.steps = steps\n        self.pipe = None\n\n    def load(self):\n        """Load the weights once, at start. The T4 has no bf16: transformer and text encoder in\n        fp16, VAE in fp32 (spec section 4). Components move to the GPU only while they run."""\n        import torch\n        from diffusers import AutoencoderKLWan, UniPCMultistepScheduler, WanPipeline\n\n        vae = AutoencoderKLWan.from_pretrained(self.path, subfolder="vae", torch_dtype=torch.float32)\n        pipe = WanPipeline.from_pretrained(self.path, vae=vae, torch_dtype=torch.float16)\n        pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=FLOW_SHIFT)\n        pipe.enable_model_cpu_offload()\n        self.pipe = pipe\n\n    def generate(self, job, inputs, should_abort, out_dir):\n        error = None\n        try:\n            return self._generate(job, should_abort, out_dir)\n        except Aborted as e:\n            # The traceback holds the pipeline\'s tensors, so drop it before freeing memory.\n            error = e.with_traceback(None)\n        finally:\n            if error is None:\n                self._free()\n        self._free()\n        raise error\n\n    def _free(self):\n        """An abort skips diffusers\' own cleanup: the transformer stayed on the GPU and the next\n        job ran out of memory (Kaggle spike). Put the model back on the CPU and drop the cache."""\n        import torch\n\n        self.pipe.maybe_free_model_hooks()\n        gc.collect()\n        torch.cuda.empty_cache()\n\n    def _generate(self, job, should_abort, out_dir):\n        import torch\n        from diffusers.utils import export_to_video\n\n        if should_abort():\n            raise Aborted()\n\n        def on_step(pipe, step, timestep, callback_kwargs):\n            if should_abort():\n                raise Aborted()          # leaves the denoising loop between two steps\n            return callback_kwargs\n\n        seed = pick_seed(job.params)\n        frames = self.pipe(\n            prompt=job.prompt,\n            negative_prompt=job.params.get("negative_prompt") or None,\n            width=WIDTH, height=HEIGHT,\n            num_frames=num_frames(job.params["duration_seconds"]),\n            num_inference_steps=self.steps,\n            guidance_scale=GUIDANCE_SCALE,\n            generator=torch.Generator("cpu").manual_seed(seed),\n            callback_on_step_end=on_step,\n        ).frames[0]\n        out = pathlib.Path(out_dir) / "clip.mp4"\n        export_to_video(frames, str(out), fps=FPS)\n        return out\n'}
root = pathlib.Path("/kaggle/working/spike")
(root / "worker").mkdir(parents=True, exist_ok=True)
for name, text in FILES.items():
    (root / "worker" / name).write_text(text)
sys.path.insert(0, str(root))
from worker.wan import WanGenerator, num_frames
from worker.errors import Aborted
print(sorted(p.name for p in (root / "worker").iterdir()))


In [ ]:
from huggingface_hub import snapshot_download
WEIGHTS = "/tmp/wan-weights/wan2-1-t2v-1-3b"      # same layout as /data/models/<model> on the instance
t = time.time()
snapshot_download("Wan-AI/Wan2.1-T2V-1.3B-Diffusers", revision="0fad780a534b6463e45facd96134c9f345acfa5b", local_dir=WEIGHTS,
                  allow_patterns=["model_index.json", "scheduler/*", "text_encoder/*", "tokenizer/*", "transformer/*", "vae/*"])
size = sum(f.stat().st_size for f in pathlib.Path(WEIGHTS).rglob("*") if f.is_file()) / 2**30
check("weights downloaded", "PASS" if 25 < size < 40 else "FAIL", f"{size:.1f} GiB in {time.time() - t:.0f} s (spec says about 29)")
os.environ["HF_HUB_OFFLINE"] = "1"                 # the image never reaches out to Hugging Face
os.environ["TRANSFORMERS_OFFLINE"] = "1"


In [ ]:
gen = WanGenerator("/tmp/wan-weights", "wan2-1-t2v-1-3b", steps=30)
t = time.time()
try:
    gen.load()
    check("model loads (fp16 transformer and text encoder, fp32 VAE, cpu offload)", "PASS",
          f"{time.time() - t:.0f} s, RAM now {psutil.Process().memory_info().rss / 2**30:.1f} GiB")
except Exception as e:
    check("model loads", "FAIL", repr(e)[:400])
    raise


In [ ]:
import types
PROMPT_A = "A cat walking through a sunlit garden, cinematic, detailed fur"
PROMPT_B = "A red balloon floating up above a city street at sunset, soft light"

import numpy as np, imageio
from PIL import Image
from IPython.display import display

def job(prompt, seconds, seed):
    return types.SimpleNamespace(prompt=prompt, params={"seed": seed, "duration_seconds": seconds, "negative_prompt": ""})

def read_frames(path):
    rd = imageio.get_reader(str(path), "ffmpeg")
    frames = [np.asarray(f) for f in rd]
    rd.close()
    return frames

def contact_sheet(frames, name):
    idx = np.linspace(0, len(frames) - 1, 6).astype(int)
    thumbs = [Image.fromarray(frames[i]).resize((416, 240)) for i in idx]
    sheet = Image.new("RGB", (416 * 3, 240 * 2))
    for n, im in enumerate(thumbs):
        sheet.paste(im, ((n % 3) * 416, (n // 3) * 240))
    sheet.save(f"/kaggle/working/{name}.png")
    display(sheet)

def quality(frames):
    a = np.stack(frames).astype(np.float32)
    motion = float(np.abs(np.diff(a, axis=0)).mean())
    return {"mean": round(float(a.mean()), 1), "std": round(float(a.std()), 1), "motion": round(motion, 2),
            "black_frames": int(sum(f.max() < 8 for f in frames))}

def run(label, prompt, seconds, steps, seed, should_abort=lambda: False, show=True):
    """One generation. Returns (frames, seconds, peak_vram_gib) or raises."""
    gen.steps = steps
    out = pathlib.Path(f"/tmp/out-{label}"); out.mkdir(exist_ok=True)
    torch.cuda.reset_peak_memory_stats()
    t = time.time()
    path = gen.generate(job(prompt, seconds, seed), {}, should_abort, out)
    secs = time.time() - t
    peak = torch.cuda.max_memory_allocated() / 2**30
    frames = read_frames(path)
    if show:
        contact_sheet(frames, label)
    return frames, secs, peak

def judge(label, frames, secs, peak, want_frames, limit_s=900):
    q = quality(frames)
    clean = q["black_frames"] == 0 and q["std"] > 10 and q["mean"] > 8 and not np.isnan(q["mean"])
    moving = q["motion"] > 0.5
    ok = len(frames) == want_frames and clean and moving and secs <= limit_s
    why = f"{len(frames)} frames (want {want_frames}), {secs:.0f} s, peak VRAM {peak:.1f} GiB (contact sheet attached), quality {q}"
    if not clean: why += " | BLACK/FLAT FRAMES: try fp32 transformer (spec section 5)"
    if not moving: why += " | FRAMES BARELY MOVE"
    if secs > limit_s: why += f" | over {limit_s} s"
    check(label, "PASS" if ok else "FAIL", why)


In [ ]:
PROMPT = "A cat walking through a sunlit garden, cinematic, detailed fur"

def timed(label, seconds="3", steps=5, width=832, height=480, guidance=5.0, embeds=None, seed=1234):
    """Run the pipeline for a few steps. Returns (seconds per step, total seconds, peak VRAM GiB, frames)."""
    from worker.wan import num_frames
    stamps = []
    def on_step(pipe, i, t, kw):
        stamps.append(time.time())
        return kw
    kwargs = dict(width=width, height=height, num_frames=num_frames(seconds), num_inference_steps=steps,
                  guidance_scale=guidance, generator=torch.Generator("cpu").manual_seed(seed),
                  callback_on_step_end=on_step, output_type="np")
    kwargs.update(embeds if embeds else {"prompt": PROMPT})
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    out = gen.pipe(**kwargs)
    total = time.time() - t0
    gaps = [b - a for a, b in zip(stamps, stamps[1:])]      # skip the first step, it includes warm-up
    per_step = sorted(gaps)[len(gaps) // 2] if gaps else float("nan")
    peak = torch.cuda.max_memory_allocated() / 2**30
    check(label, "INFO", f"{per_step:.1f} s per step, {total:.0f} s total for {steps} steps, peak VRAM {peak:.1f} GiB")
    return per_step, total, peak, out.frames[0]


## A. Baseline

In [ ]:
# A. baseline: exactly what the worker does now (enable_model_cpu_offload)
try:
    base_step, *_ = timed("A baseline: model cpu offload, 832x480, with guidance")
except Exception as e:
    check("A baseline", "FAIL", repr(e)[:300])


## B. Smaller picture

In [ ]:
# B. smaller picture: 640x368 has about 0.6x the tokens, attention cost drops faster than that
try:
    timed("B 640x368, with guidance", width=640, height=368)
except Exception as e:
    check("B 640x368", "FAIL", repr(e)[:300])


## C. No guidance

In [ ]:
# C. no classifier-free guidance (guidance_scale 1): one model pass per step instead of two
try:
    timed("C guidance_scale 1.0 (no CFG), 832x480", guidance=1.0)
except Exception as e:
    check("C no CFG", "FAIL", repr(e)[:300])


## D. Transformer stays on the GPU

In [ ]:
# D. transformer and VAE stay on the GPU; only the text encoder is moved out after encoding.
# (The whole pipeline does not fit: the text encoder alone is about 11 GiB in fp16.)
try:
    gen.pipe.remove_all_hooks()
    gen.pipe.text_encoder.to("cuda")
    with torch.no_grad():
        pe, ne = gen.pipe.encode_prompt(PROMPT, negative_prompt=None, do_classifier_free_guidance=True, device="cuda")
    gen.pipe.text_encoder.to("cpu"); torch.cuda.empty_cache()
    gen.pipe.transformer.to("cuda"); gen.pipe.vae.to("cuda")
    timed("D resident transformer and VAE, 832x480, with guidance", embeds={"prompt_embeds": pe, "negative_prompt_embeds": ne})
except Exception as e:
    check("D resident", "FAIL", repr(e)[:300])


## Report

In [ ]:
print("=== SPEED REPORT ===")
for r in REPORT:
    print(f"{r['status']:5} | {r['check']} | {r['detail']}")
json.dump(REPORT, open("/kaggle/working/speed_report.json", "w"), indent=1)


## Optional: quality at the smaller picture

In [ ]:
# OPTIONAL, about 10 minutes: does 640x368 still look fine at 20 steps? Compare with clip-3s-20steps.png from the first run.
try:
    gen.pipe.enable_model_cpu_offload()
    _, secs, peak, frames = timed("E 640x368 at 20 steps", steps=20, width=640, height=368)
    from diffusers.utils import export_to_video
    frames = [(f * 255).astype("uint8") for f in frames]
    contact_sheet(frames, "clip-640x368-20steps")
except Exception as e:
    check("E 640x368 quality", "FAIL", repr(e)[:300])


In [ ]:
print("=== SPEED REPORT ===")
for r in REPORT:
    print(f"{r['status']:5} | {r['check']} | {r['detail']}")
json.dump(REPORT, open("/kaggle/working/speed_report.json", "w"), indent=1)
